In [ ]:
import pandas as pd

In [ ]:
trans_clean = pd.read_csv("transactions_cleaned.csv")

In [ ]:
trans_clean['transaction_date'] = pd.to_datetime(trans_clean['transaction_date'])
trans_clean['cohort_month'] = pd.to_datetime(trans_clean['cohort_month'].astype(str))

In [ ]:
trans_clean['transaction_month'] = (
    trans_clean['transaction_date']
    .dt.to_period('M')
    .dt.to_timestamp()
)

print(trans_clean.head())

In [ ]:
trans_clean['cohort_index'] = (
    (trans_clean['transaction_month'].dt.year - trans_clean['cohort_month'].dt.year) * 12
    +
    (trans_clean['transaction_month'].dt.month - trans_clean['cohort_month'].dt.month)
)
print(trans_clean.head())

In [ ]:
cohort_data = (
    trans_clean.groupby(['cohort_month', 'cohort_index'])['user_id']
    .nunique()
    .reset_index()
)

print(cohort_data.to_string(index=False))

In [ ]:
retention_matrix = cohort_data.pivot_table(
    index='cohort_month',
    columns='cohort_index',
    values='user_id'
)

retention_matrix.to_csv('cohort_retention_matrix.csv', index=True)
print(retention_matrix.to_string(index=True))

In [ ]:
retention_percentage = (
    retention_matrix
    .div(retention_matrix.iloc[:,0], axis=0)
    * 100
)

print(retention_percentage.to_string(index=True))
retention_percentage.to_csv('cohort_retention_percentage.csv', index=True)